# probe.ipynb

Layer-wise emotion probe training, evaluation, and audio representation extraction.

Set the representation index and output directories, then run the section for your selected model.

Use the setup steps in the repository README before running this notebook.


In [ ]:
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

model_id = "nvidia/audio-flamingo-3-hf"
processor = AutoProcessor.from_pretrained(model_id)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(model_id, device_map="auto")
speech_prompt = """
You are a deterministic signal analyzer.

Ignore linguistic meaning. Focus ONLY on acoustic properties such as:
- pitch variation
- intensity
- speaking rate
- prosody

Task:
Estimate emotional expressiveness of the audio.

Output format (STRICT):
{"emotion_score": float,"reason":}

Constraints:
- 0.0 = flat, monotone, neutral delivery
- 1.0 = highly expressive or emotional delivery
- No explanation
- No extra text
- Must be valid JSON
"""


In [ ]:
import librosa
audio, sr = librosa.load("/content/1_1.wav", sr=16000)
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "transcribe the audio"},
            {"type": "audio", "path": audio},
        ],
    }
]

In [ ]:
inputs = processor.apply_chat_template(
    conversation,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

In [ ]:
inputs

In [ ]:
outputs = model.generate(
    **inputs,
    max_new_tokens=200
)

decoded = processor.batch_decode(
    outputs[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(decoded)

In [ ]:
outputs = model(
    **inputs,
    output_hidden_states=True,
    return_dict=True
)

hidden = outputs.hidden_states

In [ ]:
outputs = model(**inputs)
logits = outputs.logits

In [ ]:
print(logits)

In [ ]:
import torch

In [ ]:
with torch.no_grad():
    out = model(**inputs, output_hidden_states=True)

print(len(out.hidden_states))        # number of layers
print(out.hidden_states[-1].shape)   # final representation

In [ ]:
for name, module in model.named_children():
    print(name)

In [ ]:
print(inputs.keys())
print(model)

In [ ]:
audio = inputs["input_features"].to(model.device)
mask = inputs["input_features_mask"].to(model.device)

with torch.no_grad():
    audio_out = model.audio_tower(
        input_features=audio,
        input_features_mask=mask,
        output_hidden_states=True
    )

print(len(audio_out.hidden_states))
print(audio_out.hidden_states[-1].shape)

In [ ]:
import torch

def extract_audio_layer_reps(model, inputs):

    audio = inputs["input_features"].to(model.device)
    mask = inputs["input_features_mask"].to(model.device)

    with torch.no_grad():
        out = model.audio_tower(
            input_features=audio,
            input_features_mask=mask,
            output_hidden_states=True
        )

    reps = []

    for layer_hidden in out.hidden_states:
        pooled = layer_hidden.mean(dim=1)   # mean pooling
        reps.append(pooled.cpu())

    return reps

In [ ]:
extract_audio_layer_reps(model, inputs)

In [ ]:
from io import BytesIO
from urllib.request import urlopen
import librosa
from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

processor = AutoProcessor.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct", device_map="auto")
model = Qwen2AudioForConditionalGeneration.from_pretrained("Qwen/Qwen2-Audio-7B-Instruct", device_map="auto")





In [ ]:
speech_prompt = """
You are a deterministic signal analyzer.

Ignore linguistic meaning. Focus ONLY on acoustic properties such as:
- pitch variation
- intensity
- speaking rate
- prosody

Task:
Estimate emotional expressiveness of the audio.

Output format (STRICT):
{"emotion_score": float}

Constraints:
- 0.0 = flat, monotone, neutral delivery
- 1.0 = highly expressive or emotional delivery
- No explanation
- No extra text
- Must be valid JSON
"""
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": speech_prompt},
            {"type": "audio", "path": "https://huggingface.co/datasets/nvidia/AudioSkills/resolve/main/assets/WhDJDIviAOg_120_10.mp3"},
        ],
    }
]
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios = []
for message in conversation:
    if isinstance(message["content"], list):
        for ele in message["content"]:
            if ele["type"] == "audio":
                audios.append(librosa.load(
                    BytesIO(urlopen(ele['path']).read()),
                    sr=processor.feature_extractor.sampling_rate)[0]
                )

inputs = processor(text=text, audios=audios, return_tensors="pt", padding=True)
inputs.input_ids = inputs.input_ids.to("cuda")

In [ ]:
import librosa

generate_kwargs = {
    "max_new_tokens": 128,
    "do_sample": False,
    "eos_token_id": processor.tokenizer.eos_token_id,
}

abs_path = "/content/1_1.wav"

waveform, _ = librosa.load(
    abs_path,
    sr=processor.feature_extractor.sampling_rate
)

conversation = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio_url": f"file://{abs_path}"},
            {"type": "text", "text": "Transcribe the speech into Cantonese. Output only the transcription."},
        ],
    }
]

text = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=False
)

inputs = processor(
    text=text,
    audio=[waveform],
    sampling_rate=processor.feature_extractor.sampling_rate,  # 🔥 FIX
    return_tensors="pt",
    padding=True
)

inputs = {k: v.to("cuda") for k, v in inputs.items()}

generate_ids = model.generate(
    **inputs,
    **generate_kwargs
)

generate_ids = generate_ids[:, inputs["input_ids"].size(1):]

response = processor.batch_decode(
    generate_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print(response)

In [ ]:
import librosa

audio, sr = librosa.load("/content/1_1.wav", sr=16000)
conversation = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "transcribe the audio using cantonese"},
            {"type": "audio", "audio": audio},  # 🔥 must be array, not path
        ],
    }
]
text = processor.apply_chat_template(
    conversation,
    tokenize=False,   # 🔥 critical
    add_generation_prompt=True
)
inputs = processor(
    text=text,
    audios=audio,          # 🔥 correct for Qwen2-Audio
    sampling_rate=16000,     # ⚠️ fixes Whisper warning
    return_tensors="pt",
    padding=True
)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
outputs = model.generate(
    **inputs,
    **generate_kwargs
)
response = processor.batch_decode(
    outputs[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print(response)

In [ ]:
import torch
with torch.no_grad():
    outputs = model(
        **inputs,
        output_hidden_states=True,
        return_dict=True
    )

print(len(outputs.hidden_states))
print(outputs.hidden_states[-1].shape)

In [ ]:
with torch.no_grad():
    generate_ids = model.generate(
        **inputs,
        do_sample=False,
        temperature=0.7
    )

# Remove input tokens from output
generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]

# Decode to text
response = processor.batch_decode(
    generate_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print(">>> Response:", response)

In [ ]:
inputs

In [ ]:
import torch

def extract_audio_layer_reps(model, inputs):

    audio = inputs["input_features"].to(model.device)
    mask = inputs["feature_attention_mask"].to(model.device)

    with torch.no_grad():
        out = model.audio_tower(
            input_features=audio,
            input_features_mask=mask,
            output_hidden_states=True
        )

    reps = []

    for layer_hidden in out.hidden_states:
        pooled = layer_hidden.mean(dim=1)   # mean pooling
        reps.append(pooled.cpu())

    return reps

In [ ]:
extract_audio_layer_reps(model, inputs)

In [ ]:
for name, module in model.named_children():
    print(name)

In [ ]:
for name, module in model.named_modules():
    if "audio" in name.lower() or "speech" in name.lower():
        print(name)

In [ ]:
!pip uninstall transformers
!pip install git+https://github.com/huggingface/transformers@v4.51.3-Qwen2.5-Omni-preview
!pip install accelerate


In [ ]:
pip install qwen-omni-utils[decord] -U

In [ ]:
import soundfile as sf

from transformers import Qwen2_5OmniForConditionalGeneration, Qwen2_5OmniProcessor
from qwen_omni_utils import process_mm_info

# default: Load the model on the available device(s)
model = Qwen2_5OmniForConditionalGeneration.from_pretrained("Qwen/Qwen2.5-Omni-7B", torch_dtype="auto", device_map="auto")

# We recommend enabling flash_attention_2 for better acceleration and memory saving.
# model = Qwen2_5OmniForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2.5-Omni-7B",
#     torch_dtype="auto",
#     device_map="auto",
#     attn_implementation="flash_attention_2",
# )

processor = Qwen2_5OmniProcessor.from_pretrained("Qwen/Qwen2.5-Omni-7B")



In [ ]:

conversation = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are Qwen, a virtual human developed by the Qwen Team, Alibaba Group, capable of perceiving auditory and visual inputs, as well as generating text and speech."}
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "audio", "audio": "https://huggingface.co/datasets/nvidia/AudioSkills/resolve/main/assets/WhDJDIviAOg_120_10.mp3"},
        ],
    },
]

# set use audio in video
USE_AUDIO_IN_VIDEO = False

# Preparation for inference
text = processor.apply_chat_template(conversation, add_generation_prompt=True, tokenize=False)
audios, images, videos= process_mm_info(conversation, use_audio_in_video=USE_AUDIO_IN_VIDEO)


In [ ]:
audios

In [ ]:
inputs = processor(text=text, audio=audios, images=images, videos=videos, return_tensors="pt", padding=True, use_audio_in_video=USE_AUDIO_IN_VIDEO)
inputs = inputs.to(model.device).to(model.dtype)

# Inference: Generation of the output text and audio
text_ids= model.generate(**inputs, return_audio=False, use_audio_in_video=USE_AUDIO_IN_VIDEO)

text = processor.batch_decode(text_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False)
print(text)

In [ ]:
import torch
with torch.no_grad():
    generate_ids = model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        temperature=0.7,return_audio=False, use_audio_in_video=USE_AUDIO_IN_VIDEO
    )

# Remove input tokens from output
generate_ids = generate_ids[:, inputs['input_ids'].shape[1]:]

# Decode to text
response = processor.batch_decode(
    generate_ids,
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False
)[0]

print(">>> Response:", response)

In [ ]:
for name, module in model.named_children():
    print(name)

In [ ]:
for name, module in model.named_modules():
    if "audio" in name.lower() or "speech" in name.lower():
        print(name)

In [ ]:
import torch

def inspect_pt(path, verbose=False):

    data = torch.load(path, map_location="cpu")

    print(f"\n📦 File: {path}")

    # If wrapped dict (recommended format)
    if isinstance(data, dict) and "reps" in data:
        reps = data["reps"]
        print("Contains metadata:")
        for k in data:
            if k != "reps":
                print(f"  {k}: {data[k]}")
    else:
        reps = data

    print("\n🔑 Keys:")
    for k in reps.keys():
        print(f"  - {k}")

    print("\n📊 Tensor Info:")
    for k, v in reps.items():

        if not isinstance(v, torch.Tensor):
            print(f"{k}: ❌ Not a tensor ({type(v)})")
            continue

        print(f"{k}: shape={tuple(v.shape)}, dtype={v.dtype}")

        if verbose:
            print(f"   mean={v.mean().item():.4f}, std={v.std().item():.4f}")

            # print small tensors fully
            if v.numel() < 20:
                print(f"   values={v}")

    print("\n✅ Done.\n")

In [ ]:
bad_files = []

for f in os.listdir(REP_DIR):
    if not f.endswith(".pt"):
        continue
    path = os.path.join(REP_DIR, f)
    try:
        _ = torch.load(path, map_location="cpu")
    except Exception as e:
        bad_files.append((f, str(e)))

print(f"Bad files: {len(bad_files)}")
print(bad_files[:10])

In [ ]:
inspect_pt("/content/drive/MyDrive/probe/probe_representations_2/500.pt")

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

from tqdm import tqdm

# =========================
# CONFIG
# =========================
REP_DIR = "/content/drive/MyDrive/probe/probe_representations_flamingo_2"
True_REP_DIR = "/content/drive/MyDrive/probe/probe_representations_flamingo_2"
META_CSV = "/content/drive/MyDrive/probe/probe_representations_flamingo_2/probe_rep_index.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import shutil

LOCAL_REP_DIR = "/content/reps_local"

if not os.path.exists(LOCAL_REP_DIR):
    shutil.copytree(REP_DIR, LOCAL_REP_DIR)

REP_DIR = LOCAL_REP_DIR

# =========================
# AUTO-DETECT ALL LAYERS
# =========================
def get_all_layers(rep_dir, num_samples=5):
    """
    Automatically detect all layer keys from representation files
    """
    all_layers = set()

    # Find all .pt files
    pt_files = [
        f for f in os.listdir(rep_dir)
        if f.endswith('.pt') and not f.startswith('probe_')
    ]

    if not pt_files:
        raise ValueError(f"No .pt files found in {rep_dir}")

    # Sample a few files to collect layer keys
    sample_files = pt_files[:min(num_samples, len(pt_files))]

    for filename in sample_files:
        filepath = os.path.join(rep_dir, filename)
        try:
            rep = torch.load(filepath, map_location="cpu")
            all_layers.update(rep.keys())
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            continue

    # Sort layers naturally (audio_0, audio_1, ..., projector, lm_0, etc.)
    def natural_sort_key(layer):
        # Extract prefix and number for sorting
        if '_' in layer:
            prefix, num = layer.rsplit('_', 1)
            try:
                return (prefix, int(num))
            except:
                return (layer, 0)
        return (layer, 0)

    sorted_layers = sorted(list(all_layers), key=natural_sort_key)

    print(f"Auto-detected {len(sorted_layers)} layers:")
    print(sorted_layers)

    return sorted_layers

# =========================
# LABEL MAP
# =========================
EMOTION_MAP = {
    "ANG": 0,
    "DIS": 1,
    "FEA": 2,
    "HAP": 3,
    "SAD": 4,
    "NEU": 5,
    "SUR": 6
}

# =========================
# DATASET
# =========================
# =========================
# DATASET WITH CONSISTENT POOLING
# =========================
class ProbeDataset(Dataset):
    def __init__(self, df, rep_dir, layer_key):
        self.df = df
        self.rep_dir = rep_dir
        self.layer_key = layer_key

        # Determine if this is an LM layer (needs pooling)
        self.needs_pooling = layer_key.startswith('lm_')

        # filter missing reps
        self.df = self.df[
            self.df["id"].apply(
                lambda x: os.path.exists(os.path.join(rep_dir, f"{x}.pt"))
            )
        ].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rep_path = os.path.join(self.rep_dir, f"{row['id']}.pt")

        try:
            rep = torch.load(rep_path, map_location="cpu")
        except:
            # Skip corrupted files
            return self.__getitem__(np.random.randint(0, len(self.df)))

        x = rep[self.layer_key].float()

        # Apply mean pooling for LM layers (3D tensors)
        if self.needs_pooling and x.dim() == 3:
            # x shape: [batch, seq_len, hidden_dim] -> [batch, hidden_dim]
            x = x.mean(dim=1)
        elif self.needs_pooling and x.dim() == 2:
            # Already pooled, just pass through
            pass

        # Squeeze any remaining singleton dimensions
        x = x.squeeze()

        # Handle case where x might still be 2D (batch dimension)
        if x.dim() == 2:
            # Take first item if batch dimension exists
            x = x[0] if x.shape[0] == 1 else x

        # Normalize (important for probe stability)
        x = (x - x.mean()) / (x.std() + 1e-6)

        y = EMOTION_MAP[row["emotion"]]

        return x, y

# =========================
# MODEL
# =========================
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes=7):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# =========================
# TRAIN FUNCTION
# =========================
def train_probe(dataset, layer_key, epochs=10, batch_size=16):

    indices = list(range(len(dataset)))
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=42
    )

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        Subset(dataset, test_idx),
        batch_size=batch_size,
        shuffle=False
    )

    # infer input dim
    sample_x, _ = dataset[0]
    input_dim = sample_x.shape[-1]

    model = LinearProbe(input_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # -------- TRAIN --------
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[{layer_key}] Epoch {epoch}: loss={total_loss:.4f}")

    # -------- EVAL --------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(dim=-1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = correct / total
    save_path = os.path.join(True_REP_DIR, f"probe_{layer_key}.pt")
    torch.save({
        "layer": layer_key,
        "input_dim": input_dim,
        "state_dict": model.state_dict()
    }, save_path)
    print(f"Saved probe: {save_path}")
    print(f"[{layer_key}] Test Accuracy: {acc:.4f}")

    return acc

# =========================
# MAIN
# =========================
df = pd.read_csv(META_CSV)

# Auto-detect all layers instead of using hardcoded list
LAYER_KEYS = get_all_layers(REP_DIR)

results = {}

for layer in tqdm(LAYER_KEYS, desc="Probing layers"):
    print(f"\n===== Probing {layer} =====")

    dataset = ProbeDataset(df, REP_DIR, layer)

    if len(dataset) == 0:
        print(f"Skipping {layer} (no data)")
        continue

    acc = train_probe(dataset, layer)
    results[layer] = acc

# =========================
# SAVE RESULTS
# =========================
results_path = os.path.join(True_REP_DIR, "probe_results_all_layers.csv")
pd.DataFrame([
    {"layer": k, "accuracy": v} for k, v in results.items()
]).to_csv(results_path, index=False)

print("\nFinal Results:")
for layer, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{layer}: {acc:.4f}")
print(f"\nSaved to: {results_path}")

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import os
import shutil

REP_DIR_DRIVE = "/content/drive/MyDrive/probe/probe_representations_flamingo_2"
LOCAL_REP_DIR = "/content/reps_local"

# =========================
# COPY FROM DRIVE → LOCAL (SAFE)
# =========================
if not os.path.exists(LOCAL_REP_DIR):
    print("Copying representations to local disk (this will take a bit)...")
    shutil.copytree(REP_DIR_DRIVE, LOCAL_REP_DIR)
    print("Copy complete.")
else:
    print("Local copy already exists. Skipping copy.")

# 🔥 IMPORTANT: use LOCAL path from now on
REP_DIR = LOCAL_REP_DIR
SAVE_DIR = REP_DIR_DRIVE   # still save results to Drive
META_CSV = os.path.join(REP_DIR_DRIVE, "probe_rep_index.csv")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXTRA_SEEDS = [1, 2]

# =========================
# LOAD ORIGINAL RESULTS
# =========================
original_path = os.path.join(SAVE_DIR, "probe_results_all_layers.csv")

if not os.path.exists(original_path):
    raise ValueError("Run baseline first.")

original_df = pd.read_csv(original_path)
original_results = dict(zip(original_df["layer"], original_df["accuracy"]))

print("Loaded baseline.")

# =========================
# GET LAYERS
# =========================
def get_all_layers(rep_dir, num_samples=5):
    all_layers = set()
    pt_files = [
        f for f in os.listdir(rep_dir)
        if f.endswith('.pt') and not f.startswith('probe_')
    ]

    for f in pt_files[:num_samples]:
        try:
            rep = torch.load(os.path.join(rep_dir, f), map_location="cpu")
            all_layers.update(rep.keys())
        except:
            continue

    def sort_key(x):
        if "_" in x:
            p, n = x.rsplit("_", 1)
            try:
                return (p, int(n))
            except:
                pass
        return (x, 0)

    return sorted(list(all_layers), key=sort_key)

LAYER_KEYS = get_all_layers(REP_DIR)

# =========================
# LABEL MAP
# =========================
EMOTION_MAP = {
    "ANG": 0, "DIS": 1, "FEA": 2,
    "HAP": 3, "SAD": 4, "NEU": 5, "SUR": 6
}

# =========================
# GLOBAL CACHE
# =========================
GLOBAL_REP_CACHE = {}

# =========================
# DATASET
# =========================
class ProbeDataset(Dataset):
    def __init__(self, df, rep_dir, layer_key):
        self.df = df.reset_index(drop=True)
        self.rep_dir = rep_dir
        self.layer_key = layer_key
        self.needs_pooling = layer_key.startswith('lm_')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        for _ in range(5):  # retry up to 5 times
            row = self.df.iloc[idx]
            rep_path = os.path.join(self.rep_dir, f"{row['id']}.pt")

            try:
                if rep_path not in GLOBAL_REP_CACHE:
                    GLOBAL_REP_CACHE[rep_path] = torch.load(rep_path, map_location="cpu")

                rep = GLOBAL_REP_CACHE[rep_path]

                if self.layer_key not in rep:
                    raise KeyError

                x = rep[self.layer_key].float()

                if self.needs_pooling and x.dim() == 3:
                    x = x.mean(dim=1)

                x = x.squeeze()
                if x.dim() == 2:
                    x = x[0]

                x = (x - x.mean()) / (x.std() + 1e-6)
                y = EMOTION_MAP[row["emotion"]]

                return x, y

            except:
                idx = np.random.randint(0, len(self.df))

        raise RuntimeError("Too many failed samples")

# =========================
# MODEL
# =========================
class LinearProbe(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 7)

    def forward(self, x):
        return self.linear(x)

# =========================
# TRAIN FUNCTION
# =========================
def train_probe(dataset, layer_key, seed, epochs=10):

    indices = list(range(len(dataset)))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=seed)

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=64,
        shuffle=True,
        num_workers=0,  # IMPORTANT: shared cache works only with 0
        pin_memory=True
    )

    test_loader = DataLoader(
        Subset(dataset, test_idx),
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    # safe sample
    for i in range(len(dataset)):
        try:
            sample_x, _ = dataset[i]
            break
        except:
            continue

    model = LinearProbe(sample_x.shape[-1]).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            loss = criterion(model(x), y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # eval
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            pred = model(x).argmax(dim=-1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    return correct / total

# =========================
# RUN
# =========================
df = pd.read_csv(META_CSV)
all_new_results = {}

for seed in EXTRA_SEEDS:
    print(f"\n===== Seed {seed} =====")
    seed_results = {}

    for layer in tqdm(LAYER_KEYS):
        dataset = ProbeDataset(df, REP_DIR, layer)

        if len(dataset) == 0:
            continue

        acc = train_probe(dataset, layer, seed)
        seed_results[layer] = acc

    save_path = os.path.join(SAVE_DIR, f"probe_results_seed_{seed}.csv")
    pd.DataFrame([{"layer": k, "accuracy": v} for k, v in seed_results.items()]).to_csv(save_path, index=False)

    print(f"Saved: {save_path}")
    all_new_results[seed] = seed_results

# =========================
# ANALYSIS
# =========================
print("\n===== FINAL ANALYSIS =====")

aggregate = {}

for layer in LAYER_KEYS:
    vals = []

    if layer in original_results:
        vals.append(original_results[layer])

    for seed in EXTRA_SEEDS:
        if layer in all_new_results[seed]:
            vals.append(all_new_results[seed][layer])

    if vals:
        aggregate[layer] = {
            "mean": np.mean(vals),
            "std": np.std(vals),
            "vals": vals
        }

sorted_layers = sorted(aggregate.items(), key=lambda x: x[1]["mean"], reverse=True)

for layer, stats in sorted_layers:
    print(f"{layer}: {stats['mean']:.4f} ± {stats['std']:.4f} | {stats['vals']}")

print("\n===== TOP-K STABILITY =====")

K = 5
top_mean = [l for l, _ in sorted_layers[:K]]
print("Top-K (mean):", top_mean)

def get_top_k(res):
    return [l for l, _ in sorted(res.items(), key=lambda x: x[1], reverse=True)[:K]]

print("Original:", get_top_k(original_results))
for seed in EXTRA_SEEDS:
    print(f"Seed {seed}:", get_top_k(all_new_results[seed]))

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import os
import shutil

# REP_DIR_DRIVE = "/content/drive/MyDrive/probe/probe_representations_flamingo_2"
# LOCAL_REP_DIR = "/content/reps_local"
REP_DIR_DRIVE = "/content/drive/MyDrive/probe/probe_representations_2"
LOCAL_REP_DIR = "/content/reps_local_qwen"
# =========================
# COPY FROM DRIVE → LOCAL (SAFE)
# =========================
if not os.path.exists(LOCAL_REP_DIR):
    print("Copying representations to local disk (this will take a bit)...")
    shutil.copytree(REP_DIR_DRIVE, LOCAL_REP_DIR)
    print("Copy complete.")
else:
    print("Local copy already exists. Skipping copy.")

# 🔥 IMPORTANT: use LOCAL path from now on
REP_DIR = LOCAL_REP_DIR
SAVE_DIR = REP_DIR_DRIVE   # still save results to Drive
META_CSV = os.path.join(REP_DIR_DRIVE, "probe_rep_index.csv")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EXTRA_SEEDS = [1, 2]

# =========================
# LOAD ORIGINAL RESULTS
# =========================
original_path = os.path.join(SAVE_DIR, "probe_results_all_layers.csv")

if not os.path.exists(original_path):
    raise ValueError("Run baseline first.")

original_df = pd.read_csv(original_path)
original_results = dict(zip(original_df["layer"], original_df["accuracy"]))

print("Loaded baseline.")

# =========================
# GET LAYERS
# =========================
def get_all_layers(rep_dir, num_samples=5):
    all_layers = set()
    pt_files = [
        f for f in os.listdir(rep_dir)
        if f.endswith('.pt') and not f.startswith('probe_')
    ]

    for f in pt_files[:num_samples]:
        try:
            rep = torch.load(os.path.join(rep_dir, f), map_location="cpu")
            all_layers.update(rep.keys())
        except:
            continue

    def sort_key(x):
        if "_" in x:
            p, n = x.rsplit("_", 1)
            try:
                return (p, int(n))
            except:
                pass
        return (x, 0)

    return sorted(list(all_layers), key=sort_key)

LAYER_KEYS = get_all_layers(REP_DIR)

# =========================
# LABEL MAP
# =========================
EMOTION_MAP = {
    "ANG": 0, "DIS": 1, "FEA": 2,
    "HAP": 3, "SAD": 4, "NEU": 5, "SUR": 6
}

# =========================
# GLOBAL CACHE
# =========================
GLOBAL_REP_CACHE = {}

# =========================
# DATASET
# =========================
class ProbeDataset(Dataset):
    def __init__(self, df, rep_dir, layer_key):
        self.df = df.reset_index(drop=True)
        self.rep_dir = rep_dir
        self.layer_key = layer_key
        self.needs_pooling = layer_key.startswith('lm_')

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        for _ in range(5):  # retry up to 5 times
            row = self.df.iloc[idx]
            rep_path = os.path.join(self.rep_dir, f"{row['id']}.pt")

            try:
                if rep_path not in GLOBAL_REP_CACHE:
                    GLOBAL_REP_CACHE[rep_path] = torch.load(rep_path, map_location="cpu")

                rep = GLOBAL_REP_CACHE[rep_path]

                if self.layer_key not in rep:
                    raise KeyError

                x = rep[self.layer_key].float()

                if self.needs_pooling and x.dim() == 3:
                    x = x.mean(dim=1)

                x = x.squeeze()
                if x.dim() == 2:
                    x = x[0]

                x = (x - x.mean()) / (x.std() + 1e-6)
                y = EMOTION_MAP[row["emotion"]]

                return x, y

            except:
                idx = np.random.randint(0, len(self.df))

        raise RuntimeError("Too many failed samples")

# =========================
# MODEL
# =========================
class LinearProbe(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 7)

    def forward(self, x):
        return self.linear(x)

# =========================
# TRAIN FUNCTION
# =========================
def train_probe(dataset, layer_key, seed, epochs=10):

    indices = list(range(len(dataset)))
    train_idx, test_idx = train_test_split(indices, test_size=0.2, random_state=seed)

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=64,
        shuffle=True,
        num_workers=0,  # IMPORTANT: shared cache works only with 0
        pin_memory=True
    )

    test_loader = DataLoader(
        Subset(dataset, test_idx),
        batch_size=64,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )

    # safe sample
    for i in range(len(dataset)):
        try:
            sample_x, _ = dataset[i]
            break
        except:
            continue

    model = LinearProbe(sample_x.shape[-1]).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    for _ in range(epochs):
        model.train()
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)

            loss = criterion(model(x), y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # eval
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            pred = model(x).argmax(dim=-1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    return correct / total

# =========================
# RUN
# =========================
df = pd.read_csv(META_CSV)
all_new_results = {}

for seed in EXTRA_SEEDS:
    print(f"\n===== Seed {seed} =====")
    seed_results = {}

    for layer in tqdm(LAYER_KEYS):
        dataset = ProbeDataset(df, REP_DIR, layer)

        if len(dataset) == 0:
            continue

        acc = train_probe(dataset, layer, seed)
        seed_results[layer] = acc

    save_path = os.path.join(SAVE_DIR, f"probe_results_seed_{seed}.csv")
    pd.DataFrame([{"layer": k, "accuracy": v} for k, v in seed_results.items()]).to_csv(save_path, index=False)

    print(f"Saved: {save_path}")
    all_new_results[seed] = seed_results

# =========================
# ANALYSIS
# =========================
print("\n===== FINAL ANALYSIS =====")

aggregate = {}

for layer in LAYER_KEYS:
    vals = []

    if layer in original_results:
        vals.append(original_results[layer])

    for seed in EXTRA_SEEDS:
        if layer in all_new_results[seed]:
            vals.append(all_new_results[seed][layer])

    if vals:
        aggregate[layer] = {
            "mean": np.mean(vals),
            "std": np.std(vals),
            "vals": vals
        }

sorted_layers = sorted(aggregate.items(), key=lambda x: x[1]["mean"], reverse=True)

for layer, stats in sorted_layers:
    print(f"{layer}: {stats['mean']:.4f} ± {stats['std']:.4f} | {stats['vals']}")

print("\n===== TOP-K STABILITY =====")

K = 5
top_mean = [l for l, _ in sorted_layers[:K]]
print("Top-K (mean):", top_mean)

def get_top_k(res):
    return [l for l, _ in sorted(res.items(), key=lambda x: x[1], reverse=True)[:K]]

print("Original:", get_top_k(original_results))
for seed in EXTRA_SEEDS:
    print(f"Seed {seed}:", get_top_k(all_new_results[seed]))

In [ ]:
import os
import torch
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split

from tqdm import tqdm
# =========================
# CONFIG
# =========================
REP_DIR = "/content/drive/MyDrive/probe/probe_representations_2"
True_REP_DIR = "/content/drive/MyDrive/probe/probe_representations_2"
META_CSV = "/content/drive/MyDrive/probe/probe_representations_2/probe_rep_index.csv"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

import shutil

LOCAL_REP_DIR = "/content/reps_local_qwen"


if not os.path.exists(LOCAL_REP_DIR):
    shutil.copytree(REP_DIR, LOCAL_REP_DIR)

REP_DIR = LOCAL_REP_DIR

# =========================
# AUTO-DETECT ALL LAYERS
# =========================
def get_all_layers(rep_dir, num_samples=5):
    """
    Automatically detect all layer keys from representation files
    """
    all_layers = set()

    # Find all .pt files
    pt_files = [f for f in os.listdir(rep_dir) if f.endswith('.pt')]

    if not pt_files:
        raise ValueError(f"No .pt files found in {rep_dir}")

    # Sample a few files to collect layer keys
    sample_files = pt_files[:min(num_samples, len(pt_files))]

    for filename in sample_files:
        filepath = os.path.join(rep_dir, filename)
        try:
            rep = torch.load(filepath, map_location="cpu")
            all_layers.update(rep.keys())
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            continue

    # Sort layers naturally (audio_0, audio_1, ..., projector, lm_0, etc.)
    def natural_sort_key(layer):
        # Extract prefix and number for sorting
        if '_' in layer:
            prefix, num = layer.rsplit('_', 1)
            try:
                return (prefix, int(num))
            except:
                return (layer, 0)
        return (layer, 0)

    sorted_layers = sorted(list(all_layers), key=natural_sort_key)

    print(f"Auto-detected {len(sorted_layers)} layers:")
    print(sorted_layers)

    return sorted_layers

# =========================
# LABEL MAP
# =========================
EMOTION_MAP = {
    "ANG": 0,
    "DIS": 1,
    "FEA": 2,
    "HAP": 3,
    "SAD": 4,
    "NEU": 5,
    "SUR": 6
}

# =========================
# DATASET
# =========================
class ProbeDataset(Dataset):
    def __init__(self, df, rep_dir, layer_key):
        self.df = df
        self.rep_dir = rep_dir
        self.layer_key = layer_key

        # Determine if this is an LM layer (needs pooling)
        self.needs_pooling = layer_key.startswith('lm_')

        # filter missing reps
        self.df = self.df[
            self.df["id"].apply(
                lambda x: os.path.exists(os.path.join(rep_dir, f"{x}.pt"))
            )
        ].reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        rep_path = os.path.join(self.rep_dir, f"{row['id']}.pt")

        try:
            rep = torch.load(rep_path, map_location="cpu")
        except:
            # Skip corrupted files
            return self.__getitem__(np.random.randint(0, len(self.df)))

        x = rep[self.layer_key].float()

        # Apply mean pooling for LM layers (3D tensors)
        if self.needs_pooling and x.dim() == 3:
            # x shape: [batch, seq_len, hidden_dim] -> [batch, hidden_dim]
            x = x.mean(dim=1)
        elif self.needs_pooling and x.dim() == 2:
            # Already pooled, just pass through
            pass

        # Squeeze any remaining singleton dimensions
        x = x.squeeze()

        # Handle case where x might still be 2D (batch dimension)
        if x.dim() == 2:
            # Take first item if batch dimension exists
            x = x[0] if x.shape[0] == 1 else x

        # Normalize (important for probe stability)
        x = (x - x.mean()) / (x.std() + 1e-6)

        y = EMOTION_MAP[row["emotion"]]

        return x, y


# =========================
# MODEL
# =========================
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes=7):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# =========================
# TRAIN FUNCTION
# =========================
def train_probe(dataset, layer_key, epochs=10, batch_size=16):

    indices = list(range(len(dataset)))
    train_idx, test_idx = train_test_split(
        indices, test_size=0.2, random_state=42
    )

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=batch_size,
        shuffle=True
    )

    test_loader = DataLoader(
        Subset(dataset, test_idx),
        batch_size=batch_size,
        shuffle=False
    )

    # infer input dim
    sample_x, _ = dataset[0]
    input_dim = sample_x.shape[-1]

    model = LinearProbe(input_dim).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()

    # -------- TRAIN --------
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"[{layer_key}] Epoch {epoch}: loss={total_loss:.4f}")

    # -------- EVAL --------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            pred = model(x).argmax(dim=-1)

            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = correct / total
    save_path = os.path.join(True_REP_DIR, f"probe_{layer_key}.pt")
    torch.save({
        "layer": layer_key,
        "input_dim": input_dim,
        "state_dict": model.state_dict()
    }, save_path)
    print(f"Saved probe: {save_path}")
    print(f"[{layer_key}] Test Accuracy: {acc:.4f}")

    return acc

# =========================
# MAIN
# =========================
df = pd.read_csv(META_CSV)

# Auto-detect all layers instead of using hardcoded list
LAYER_KEYS = get_all_layers(REP_DIR)

results = {}

for layer in tqdm(LAYER_KEYS, desc="Probing layers"):
    print(f"\n===== Probing {layer} =====")

    dataset = ProbeDataset(df, REP_DIR, layer)

    if len(dataset) == 0:
        print(f"Skipping {layer} (no data)")
        continue

    acc = train_probe(dataset, layer)
    results[layer] = acc

# =========================
# SAVE RESULTS
# =========================
results_path = os.path.join(True_REP_DIR, "probe_results_all_layers.csv")
pd.DataFrame([
    {"layer": k, "accuracy": v} for k, v in results.items()
]).to_csv(results_path, index=False)

print("\nFinal Results:")
for layer, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{layer}: {acc:.4f}")
print(f"\nSaved to: {results_path}")

In [ ]:
import os
import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# =========================
# CONFIG
# =========================
BASE_DIR = "/content/drive/MyDrive/probing/full_probe/albe"
#BASE_DIR = "/content/drive/MyDrive/subprobe/task3_full_run_qwen/layers"
MANIP_CSV = os.path.join(BASE_DIR, "manipulated_results.csv")
NEUTRAL_CSV = os.path.join(BASE_DIR, "neutral_results.csv")

PROBE_DIR = "/content/drive/MyDrive/probe/probe_representations_2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# AUTO-DETECT ALL LAYERS FROM PROBE FILES
# =========================
def get_all_probe_layers(probe_dir):
    """
    Automatically detect all available probe layers from saved .pt files
    """
    all_layers = []

    # Find all probe files
    probe_files = [f for f in os.listdir(probe_dir) if f.startswith('probe_') and f.endswith('.pt')]

    if not probe_files:
        raise ValueError(f"No probe files found in {probe_dir}")

    # Extract layer names from filenames
    for filename in probe_files:
        # Remove 'probe_' prefix and '.pt' suffix
        layer_name = filename.replace('probe_', '').replace('.pt', '')
        all_layers.append(layer_name)

    # Sort layers naturally
    def natural_sort_key(layer):
        if '_' in layer:
            prefix, num = layer.rsplit('_', 1)
            try:
                return (prefix, int(num))
            except:
                return (layer, 0)
        return (layer, 0)

    sorted_layers = sorted(all_layers, key=natural_sort_key)

    print(f"Auto-detected {len(sorted_layers)} probe layers:")
    print(sorted_layers)

    return sorted_layers

# =========================
# LABEL MAP
# =========================
IDX2EMO = {
    0: "ANG",
    1: "DIS",
    2: "FEA",
    3: "HAP",
    4: "SAD",
    5: "NEU",
    6: "SUR"
}

# =========================
# MODEL
# =========================
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes=7):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# =========================
# LOAD PROBES
# =========================
def load_probes(layer_keys):
    probes = {}

    for layer in layer_keys:
        path = os.path.join(PROBE_DIR, f"probe_{layer}.pt")

        if not os.path.exists(path):
            print(f"⚠️ Missing probe: {layer}")
            continue

        ckpt = torch.load(path, map_location="cpu")

        model = LinearProbe(ckpt["input_dim"])
        model.load_state_dict(ckpt["state_dict"])
        model.to(DEVICE)
        model.eval()

        probes[layer] = model

    print(f"\nLoaded {len(probes)} probes: {list(probes.keys())}")
    return probes

# =========================
# APPLY PROBE FUNCTION
# =========================
def apply_probes(input_csv, output_csv, probes, apply_pooling=True):

    df = pd.read_csv(input_csv)
    print(f"Processing {len(df)} rows from {input_csv}")

    results = []
    error_count = 0
    skipped_layers = set()

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {os.path.basename(input_csv)}"):

        rep_path = row["rep_path"]

        if not isinstance(rep_path, str) or not os.path.exists(rep_path):
            continue

        try:
            rep = torch.load(rep_path, map_location="cpu")
        except Exception as e:
            continue

        for layer, model in probes.items():

            if layer not in rep:
                if layer not in skipped_layers:
                    print(f"\n⚠️ Layer {layer} not found in representation file {rep_path}")
                    skipped_layers.add(layer)
                continue

            try:
                x = rep[layer].float()

                # Apply mean pooling for LM layers if requested
                if apply_pooling and layer.startswith('lm_') and x.dim() == 3:
                    # x shape: [1, seq_len, hidden_dim] -> [1, hidden_dim]
                    x = x.mean(dim=1)

                # Handle different tensor shapes
                if x.dim() > 1:
                    # If still 2D or 3D, flatten or squeeze appropriately
                    if x.dim() == 2 and x.shape[0] == 1:
                        x = x.squeeze(0)  # Remove batch dimension
                    elif x.dim() == 2 and x.shape[0] > 1:
                        # Multiple items - take mean
                        x = x.mean(dim=0)
                    elif x.dim() > 2:
                        # Flatten everything else
                        x = x.flatten()

                # Normalize
                x = (x - x.mean()) / (x.std() + 1e-6)

                # Add batch dimension
                x = x.unsqueeze(0).to(DEVICE)  # [1, features]

                with torch.no_grad():
                    logits = model(x)
                    probs = F.softmax(logits, dim=-1)

                    prob_vec = probs.squeeze(0).cpu().numpy()  # shape [7]
                    pred_idx = prob_vec.argmax()
                    conf = prob_vec.max()

                    entry = {
                        "neutral_path": row.get("neutral_path", None),
                        "manipulated_path": row.get("manipulated_path", None),
                        "rep_path": rep_path,
                        "layer": layer,
                        "pred": IDX2EMO[pred_idx],
                        "confidence": conf
                    }

                    # Add all probabilities
                    for i in range(7):
                        entry[f"p_{IDX2EMO[i]}"] = prob_vec[i]

                    results.append(entry)

            except Exception as e:
                error_count += 1
                if error_count <= 10:  # Show first 10 errors
                    print(f"\nError on row {idx}, layer {layer}: {e}")
                    print(f"x shape: {x.shape if 'x' in locals() else 'unknown'}")
                continue

    # Create DataFrame and save
    result_df = pd.DataFrame(results)
    result_df.to_csv(output_csv, index=False)

    print(f"\n✅ Saved: {output_csv}")
    print(f"   Total predictions: {len(results)}")
    print(f"   Unique layers: {result_df['layer'].nunique()}")
    print(f"   Total errors encountered: {error_count}")

    return result_df

# =========================
# RUN
# =========================
def main():
    # Auto-detect all layers from probe directory
    print("=" * 50)
    print("STEP 1: Detecting all probe layers")
    print("=" * 50)
    LAYER_KEYS = get_all_probe_layers(PROBE_DIR)

    print("\n" + "=" * 50)
    print("STEP 2: Loading probes")
    print("=" * 50)
    probes = load_probes(LAYER_KEYS)

    if not probes:
        print("❌ No probes loaded! Exiting.")
        return

    print("\n" + "=" * 50)
    print("STEP 3: Running probes on manipulated data")
    print("=" * 50)
    manip_results = apply_probes(
        MANIP_CSV,
        os.path.join(BASE_DIR, "manipulated_probe_predictions_all_layers.csv"),
        probes,
        apply_pooling=True
    )
    # run neutral
    apply_probes(
        NEUTRAL_CSV,
        os.path.join(BASE_DIR, "neutral_probe_predictions.csv"),
        probes,apply_pooling=True
    )

    # Optional: Print summary statistics
    print("\n" + "=" * 50)
    print("SUMMARY STATISTICS")
    print("=" * 50)

    # Layer-wise accuracy (if you have ground truth)
    if 'emotion' in manip_results.columns:
        layer_acc = manip_results.groupby('layer').apply(
            lambda x: (x['pred'] == x['emotion']).mean()
        ).sort_values(ascending=False)

        print("\nTop 10 layers by accuracy:")
        print(layer_acc.head(10))

        print("\nBottom 10 layers by accuracy:")
        print(layer_acc.tail(10))

    # Confidence statistics
    layer_conf = manip_results.groupby('layer')['confidence'].agg(['mean', 'std', 'count'])
    print("\nConfidence statistics by layer:")
    print(layer_conf.head(10))

# =========================
# RUN MAIN
# =========================
if __name__ == "__main__":
    main()



In [ ]:
import os
import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# =========================
# CONFIG
# =========================
BASE_DIR = "/content/drive/MyDrive/subprobe/task3_full_run_qwen/layers"
MANIP_CSV = os.path.join(BASE_DIR, "task3_results.csv")
PROBE_DIR = "/content/drive/MyDrive/probe/probe_representations_2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# AUTO-DETECT ALL LAYERS FROM PROBE FILES
# =========================
def get_all_probe_layers(probe_dir):
    """
    Automatically detect all available probe layers from saved .pt files
    """
    all_layers = []

    # Find all probe files
    probe_files = [f for f in os.listdir(probe_dir) if f.startswith('probe_') and f.endswith('.pt')]

    if not probe_files:
        raise ValueError(f"No probe files found in {probe_dir}")

    # Extract layer names from filenames
    for filename in probe_files:
        # Remove 'probe_' prefix and '.pt' suffix
        layer_name = filename.replace('probe_', '').replace('.pt', '')
        all_layers.append(layer_name)

    # Sort layers naturally
    def natural_sort_key(layer):
        if '_' in layer:
            prefix, num = layer.rsplit('_', 1)
            try:
                return (prefix, int(num))
            except:
                return (layer, 0)
        return (layer, 0)

    sorted_layers = sorted(all_layers, key=natural_sort_key)

    print(f"Auto-detected {len(sorted_layers)} probe layers:")
    print(sorted_layers)

    return sorted_layers

# =========================
# LABEL MAP
# =========================
IDX2EMO = {
    0: "ANG",
    1: "DIS",
    2: "FEA",
    3: "HAP",
    4: "SAD",
    5: "NEU",
    6: "SUR"
}

# =========================
# MODEL
# =========================
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes=7):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# =========================
# LOAD PROBES
# =========================
def load_probes(layer_keys):
    probes = {}

    for layer in layer_keys:
        path = os.path.join(PROBE_DIR, f"probe_{layer}.pt")

        if not os.path.exists(path):
            print(f"⚠️ Missing probe: {layer}")
            continue

        ckpt = torch.load(path, map_location="cpu")

        model = LinearProbe(ckpt["input_dim"])
        model.load_state_dict(ckpt["state_dict"])
        model.to(DEVICE)
        model.eval()

        probes[layer] = model

    print(f"\nLoaded {len(probes)} probes: {list(probes.keys())}")
    return probes

# =========================
# APPLY PROBE FUNCTION
# =========================
def apply_probes(input_csv, output_csv, probes, apply_pooling=True):

    df = pd.read_csv(input_csv)
    print(f"Processing {len(df)} rows from {input_csv}")

    results = []
    error_count = 0
    skipped_layers = set()

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {os.path.basename(input_csv)}"):

        rep_path = row["rep_path"]

        if not isinstance(rep_path, str) or not os.path.exists(rep_path):
            continue

        try:
            rep = torch.load(rep_path, map_location="cpu")
        except Exception as e:
            continue

        for layer, model in probes.items():

            if layer not in rep:
                if layer not in skipped_layers:
                    print(f"\n⚠️ Layer {layer} not found in representation file {rep_path}")
                    skipped_layers.add(layer)
                continue

            try:
                x = rep[layer].float()

                # Apply mean pooling for LM layers if requested
                if apply_pooling and layer.startswith('lm_') and x.dim() == 3:
                    # x shape: [1, seq_len, hidden_dim] -> [1, hidden_dim]
                    x = x.mean(dim=1)

                # Handle different tensor shapes
                if x.dim() > 1:
                    # If still 2D or 3D, flatten or squeeze appropriately
                    if x.dim() == 2 and x.shape[0] == 1:
                        x = x.squeeze(0)  # Remove batch dimension
                    elif x.dim() == 2 and x.shape[0] > 1:
                        # Multiple items - take mean
                        x = x.mean(dim=0)
                    elif x.dim() > 2:
                        # Flatten everything else
                        x = x.flatten()

                # Normalize
                x = (x - x.mean()) / (x.std() + 1e-6)

                # Add batch dimension
                x = x.unsqueeze(0).to(DEVICE)  # [1, features]

                with torch.no_grad():
                    logits = model(x)
                    probs = F.softmax(logits, dim=-1)

                    prob_vec = probs.squeeze(0).cpu().numpy()  # shape [7]
                    pred_idx = prob_vec.argmax()
                    conf = prob_vec.max()

                    entry = {
                        "neutral_path": row.get("neutral_path", None),
                        "manipulated_path": row.get("manipulated_path", None),
                        "rep_path": rep_path,
                        "layer": layer,
                        "pred": IDX2EMO[pred_idx],
                        "confidence": conf
                    }

                    # Add all probabilities
                    for i in range(7):
                        entry[f"p_{IDX2EMO[i]}"] = prob_vec[i]

                    results.append(entry)

            except Exception as e:
                error_count += 1
                if error_count <= 10:  # Show first 10 errors
                    print(f"\nError on row {idx}, layer {layer}: {e}")
                    print(f"x shape: {x.shape if 'x' in locals() else 'unknown'}")
                continue

    # Create DataFrame and save
    result_df = pd.DataFrame(results)
    result_df.to_csv(output_csv, index=False)

    print(f"\n✅ Saved: {output_csv}")
    print(f"   Total predictions: {len(results)}")
    print(f"   Unique layers: {result_df['layer'].nunique()}")
    print(f"   Total errors encountered: {error_count}")

    return result_df

# =========================
# RUN
# =========================
def main():
    # Auto-detect all layers from probe directory
    print("=" * 50)
    print("STEP 1: Detecting all probe layers")
    print("=" * 50)
    LAYER_KEYS = get_all_probe_layers(PROBE_DIR)

    print("\n" + "=" * 50)
    print("STEP 2: Loading probes")
    print("=" * 50)
    probes = load_probes(LAYER_KEYS)

    if not probes:
        print("❌ No probes loaded! Exiting.")
        return

    print("\n" + "=" * 50)
    print("STEP 3: Running probes on manipulated data")
    print("=" * 50)
    manip_results = apply_probes(
        MANIP_CSV,
        os.path.join(BASE_DIR, "manipulated_probe_predictions_all_layers.csv"),
        probes,
        apply_pooling=True
    )

    # Optional: Print summary statistics
    print("\n" + "=" * 50)
    print("SUMMARY STATISTICS")
    print("=" * 50)

    # Layer-wise accuracy (if you have ground truth)
    if 'emotion' in manip_results.columns:
        layer_acc = manip_results.groupby('layer').apply(
            lambda x: (x['pred'] == x['emotion']).mean()
        ).sort_values(ascending=False)

        print("\nTop 10 layers by accuracy:")
        print(layer_acc.head(10))

        print("\nBottom 10 layers by accuracy:")
        print(layer_acc.tail(10))

    # Confidence statistics
    layer_conf = manip_results.groupby('layer')['confidence'].agg(['mean', 'std', 'count'])
    print("\nConfidence statistics by layer:")
    print(layer_conf.head(10))

# =========================
# RUN MAIN
# =========================
if __name__ == "__main__":
    main()

In [ ]:
import os
import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# =========================
# CONFIG
# =========================
BASE_DIR = "/content/drive/MyDrive/probing/full_probe_flamingo/abla"
#BASE_DIR = "/content/drive/MyDrive/subprobe/task3_full_run_flamingo"
MANIP_CSV = os.path.join(BASE_DIR, "manipulation_results.csv")
NEUTRAL_CSV = os.path.join(BASE_DIR, "neutral_results.csv")
#PROBE_DIR = "/content/drive/MyDrive/probe/probe_representations"
PROBE_DIR = "/content/drive/MyDrive/probe/probe_representations_flamingo_2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# AUTO-DETECT ALL LAYERS FROM PROBE FILES
# =========================
def get_all_probe_layers(probe_dir):
    """
    Automatically detect all available probe layers from saved .pt files
    """
    all_layers = []

    # Find all probe files
    probe_files = [f for f in os.listdir(probe_dir) if f.startswith('probe_') and f.endswith('.pt')]

    if not probe_files:
        raise ValueError(f"No probe files found in {probe_dir}")

    # Extract layer names from filenames
    for filename in probe_files:
        # Remove 'probe_' prefix and '.pt' suffix
        layer_name = filename.replace('probe_', '').replace('.pt', '')
        all_layers.append(layer_name)

    # Sort layers naturally
    def natural_sort_key(layer):
        if '_' in layer:
            prefix, num = layer.rsplit('_', 1)
            try:
                return (prefix, int(num))
            except:
                return (layer, 0)
        return (layer, 0)

    sorted_layers = sorted(all_layers, key=natural_sort_key)

    print(f"Auto-detected {len(sorted_layers)} probe layers:")
    print(sorted_layers)

    return sorted_layers

# =========================
# LABEL MAP
# =========================
IDX2EMO = {
    0: "ANG",
    1: "DIS",
    2: "FEA",
    3: "HAP",
    4: "SAD",
    5: "NEU",
    6: "SUR"
}

# =========================
# MODEL
# =========================
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes=7):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# =========================
# LOAD PROBES
# =========================
def load_probes(layer_keys):
    probes = {}

    for layer in layer_keys:
        path = os.path.join(PROBE_DIR, f"probe_{layer}.pt")

        if not os.path.exists(path):
            print(f"⚠️ Missing probe: {layer}")
            continue

        ckpt = torch.load(path, map_location="cpu")

        model = LinearProbe(ckpt["input_dim"])
        model.load_state_dict(ckpt["state_dict"])
        model.to(DEVICE)
        model.eval()

        probes[layer] = model

    print(f"\nLoaded {len(probes)} probes: {list(probes.keys())}")
    return probes

# =========================
# APPLY PROBE FUNCTION
# =========================
def apply_probes(input_csv, output_csv, probes, apply_pooling=True):

    df = pd.read_csv(input_csv)
    print(f"Processing {len(df)} rows from {input_csv}")

    results = []
    error_count = 0
    skipped_layers = set()

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {os.path.basename(input_csv)}"):

        rep_path = row["rep_path"]

        if not isinstance(rep_path, str) or not os.path.exists(rep_path):
            continue

        try:
            rep = torch.load(rep_path, map_location="cpu")
        except Exception as e:
            continue

        for layer, model in probes.items():

            if layer not in rep:
                if layer not in skipped_layers:
                    print(f"\n⚠️ Layer {layer} not found in representation file {rep_path}")
                    skipped_layers.add(layer)
                continue

            try:
                x = rep[layer].float()

                # Apply mean pooling for LM layers if requested
                if apply_pooling and layer.startswith('lm_') and x.dim() == 3:
                    # x shape: [1, seq_len, hidden_dim] -> [1, hidden_dim]
                    x = x.mean(dim=1)

                # Handle different tensor shapes
                if x.dim() > 1:
                    # If still 2D or 3D, flatten or squeeze appropriately
                    if x.dim() == 2 and x.shape[0] == 1:
                        x = x.squeeze(0)  # Remove batch dimension
                    elif x.dim() == 2 and x.shape[0] > 1:
                        # Multiple items - take mean
                        x = x.mean(dim=0)
                    elif x.dim() > 2:
                        # Flatten everything else
                        x = x.flatten()

                # Normalize
                x = (x - x.mean()) / (x.std() + 1e-6)

                # Add batch dimension
                x = x.unsqueeze(0).to(DEVICE)  # [1, features]

                with torch.no_grad():
                    logits = model(x)
                    probs = F.softmax(logits, dim=-1)

                    prob_vec = probs.squeeze(0).cpu().numpy()  # shape [7]
                    pred_idx = prob_vec.argmax()
                    conf = prob_vec.max()

                    entry = {
                        "neutral_path": row.get("neutral_path", None),
                        "manipulated_path": row.get("manipulated_path", None),
                        "rep_path": rep_path,
                        "layer": layer,
                        "pred": IDX2EMO[pred_idx],
                        "confidence": conf
                    }

                    # Add all probabilities
                    for i in range(7):
                        entry[f"p_{IDX2EMO[i]}"] = prob_vec[i]

                    results.append(entry)

            except Exception as e:
                error_count += 1
                if error_count <= 10:  # Show first 10 errors
                    print(f"\nError on row {idx}, layer {layer}: {e}")
                    print(f"x shape: {x.shape if 'x' in locals() else 'unknown'}")
                continue

    # Create DataFrame and save
    result_df = pd.DataFrame(results)
    result_df.to_csv(output_csv, index=False)

    print(f"\n✅ Saved: {output_csv}")
    print(f"   Total predictions: {len(results)}")
    print(f"   Unique layers: {result_df['layer'].nunique()}")
    print(f"   Total errors encountered: {error_count}")

    return result_df

# =========================
# RUN
# =========================
def main():
    # Auto-detect all layers from probe directory
    print("=" * 50)
    print("STEP 1: Detecting all probe layers")
    print("=" * 50)
    LAYER_KEYS = get_all_probe_layers(PROBE_DIR)

    print("\n" + "=" * 50)
    print("STEP 2: Loading probes")
    print("=" * 50)
    probes = load_probes(LAYER_KEYS)

    if not probes:
        print("❌ No probes loaded! Exiting.")
        return

    print("\n" + "=" * 50)
    print("STEP 3: Running probes on manipulated data")
    print("=" * 50)
    manip_results = apply_probes(
        MANIP_CSV,
        os.path.join(BASE_DIR, "manipulation_probe_predictions_all_layers.csv"),
        probes,
        apply_pooling=True
    )
    # run neutral
    apply_probes(
        NEUTRAL_CSV,
        os.path.join(BASE_DIR, "neutral_probe_predictions.csv"),
        probes,apply_pooling=True
    )

    # Optional: Print summary statistics
    print("\n" + "=" * 50)
    print("SUMMARY STATISTICS")
    print("=" * 50)

    # Layer-wise accuracy (if you have ground truth)
    if 'emotion' in manip_results.columns:
        layer_acc = manip_results.groupby('layer').apply(
            lambda x: (x['pred'] == x['emotion']).mean()
        ).sort_values(ascending=False)

        print("\nTop 10 layers by accuracy:")
        print(layer_acc.head(10))

        print("\nBottom 10 layers by accuracy:")
        print(layer_acc.tail(10))

    # Confidence statistics
    layer_conf = manip_results.groupby('layer')['confidence'].agg(['mean', 'std', 'count'])
    print("\nConfidence statistics by layer:")
    print(layer_conf.head(10))

# =========================
# RUN MAIN
# =========================
if __name__ == "__main__":
    main()


In [ ]:
import os
import torch
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm

# =========================
# CONFIG
# =========================
#BASE_DIR = "/content/drive/MyDrive/probing/full_probe_flamingo_2"
BASE_DIR = "/content/drive/MyDrive/subprobe/task3_full_run_flamingo/layers"
MANIP_CSV = os.path.join(BASE_DIR, "task3_results.csv")
#NEUTRAL_CSV = os.path.join(BASE_DIR, "neutral_results.csv")
#PROBE_DIR = "/content/drive/MyDrive/probe/probe_representations"
PROBE_DIR = "/content/drive/MyDrive/probe/probe_representations_flamingo_2"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# AUTO-DETECT ALL LAYERS FROM PROBE FILES
# =========================
def get_all_probe_layers(probe_dir):
    """
    Automatically detect all available probe layers from saved .pt files
    """
    all_layers = []

    # Find all probe files
    probe_files = [f for f in os.listdir(probe_dir) if f.startswith('probe_') and f.endswith('.pt')]

    if not probe_files:
        raise ValueError(f"No probe files found in {probe_dir}")

    # Extract layer names from filenames
    for filename in probe_files:
        # Remove 'probe_' prefix and '.pt' suffix
        layer_name = filename.replace('probe_', '').replace('.pt', '')
        all_layers.append(layer_name)

    # Sort layers naturally
    def natural_sort_key(layer):
        if '_' in layer:
            prefix, num = layer.rsplit('_', 1)
            try:
                return (prefix, int(num))
            except:
                return (layer, 0)
        return (layer, 0)

    sorted_layers = sorted(all_layers, key=natural_sort_key)

    print(f"Auto-detected {len(sorted_layers)} probe layers:")
    print(sorted_layers)

    return sorted_layers

# =========================
# LABEL MAP
# =========================
IDX2EMO = {
    0: "ANG",
    1: "DIS",
    2: "FEA",
    3: "HAP",
    4: "SAD",
    5: "NEU",
    6: "SUR"
}

# =========================
# MODEL
# =========================
class LinearProbe(nn.Module):
    def __init__(self, input_dim, num_classes=7):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# =========================
# LOAD PROBES
# =========================
def load_probes(layer_keys):
    probes = {}

    for layer in layer_keys:
        path = os.path.join(PROBE_DIR, f"probe_{layer}.pt")

        if not os.path.exists(path):
            print(f"⚠️ Missing probe: {layer}")
            continue

        ckpt = torch.load(path, map_location="cpu")

        model = LinearProbe(ckpt["input_dim"])
        model.load_state_dict(ckpt["state_dict"])
        model.to(DEVICE)
        model.eval()

        probes[layer] = model

    print(f"\nLoaded {len(probes)} probes: {list(probes.keys())}")
    return probes

# =========================
# APPLY PROBE FUNCTION
# =========================
def apply_probes(input_csv, output_csv, probes, apply_pooling=True):

    df = pd.read_csv(input_csv)
    print(f"Processing {len(df)} rows from {input_csv}")

    results = []
    error_count = 0
    skipped_layers = set()

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {os.path.basename(input_csv)}"):

        rep_path = row["rep_path"]

        if not isinstance(rep_path, str) or not os.path.exists(rep_path):
            continue

        try:
            rep = torch.load(rep_path, map_location="cpu")
        except Exception as e:
            continue

        for layer, model in probes.items():

            if layer not in rep:
                if layer not in skipped_layers:
                    print(f"\n⚠️ Layer {layer} not found in representation file {rep_path}")
                    skipped_layers.add(layer)
                continue

            try:
                x = rep[layer].float()

                # Apply mean pooling for LM layers if requested
                if apply_pooling and layer.startswith('lm_') and x.dim() == 3:
                    # x shape: [1, seq_len, hidden_dim] -> [1, hidden_dim]
                    x = x.mean(dim=1)

                # Handle different tensor shapes
                if x.dim() > 1:
                    # If still 2D or 3D, flatten or squeeze appropriately
                    if x.dim() == 2 and x.shape[0] == 1:
                        x = x.squeeze(0)  # Remove batch dimension
                    elif x.dim() == 2 and x.shape[0] > 1:
                        # Multiple items - take mean
                        x = x.mean(dim=0)
                    elif x.dim() > 2:
                        # Flatten everything else
                        x = x.flatten()

                # Normalize
                x = (x - x.mean()) / (x.std() + 1e-6)

                # Add batch dimension
                x = x.unsqueeze(0).to(DEVICE)  # [1, features]

                with torch.no_grad():
                    logits = model(x)
                    probs = F.softmax(logits, dim=-1)

                    prob_vec = probs.squeeze(0).cpu().numpy()  # shape [7]
                    pred_idx = prob_vec.argmax()
                    conf = prob_vec.max()

                    entry = {
                        "neutral_path": row.get("neutral_path", None),
                        "manipulated_path": row.get("manipulated_path", None),
                        "rep_path": rep_path,
                        "layer": layer,
                        "pred": IDX2EMO[pred_idx],
                        "confidence": conf
                    }

                    # Add all probabilities
                    for i in range(7):
                        entry[f"p_{IDX2EMO[i]}"] = prob_vec[i]

                    results.append(entry)

            except Exception as e:
                error_count += 1
                if error_count <= 10:  # Show first 10 errors
                    print(f"\nError on row {idx}, layer {layer}: {e}")
                    print(f"x shape: {x.shape if 'x' in locals() else 'unknown'}")
                continue

    # Create DataFrame and save
    result_df = pd.DataFrame(results)
    result_df.to_csv(output_csv, index=False)

    print(f"\n✅ Saved: {output_csv}")
    print(f"   Total predictions: {len(results)}")
    print(f"   Unique layers: {result_df['layer'].nunique()}")
    print(f"   Total errors encountered: {error_count}")

    return result_df

# =========================
# RUN
# =========================
def main():
    # Auto-detect all layers from probe directory
    print("=" * 50)
    print("STEP 1: Detecting all probe layers")
    print("=" * 50)
    LAYER_KEYS = get_all_probe_layers(PROBE_DIR)

    print("\n" + "=" * 50)
    print("STEP 2: Loading probes")
    print("=" * 50)
    probes = load_probes(LAYER_KEYS)

    if not probes:
        print("❌ No probes loaded! Exiting.")
        return

    print("\n" + "=" * 50)
    print("STEP 3: Running probes on manipulated data")
    print("=" * 50)
    manip_results = apply_probes(
        MANIP_CSV,
        os.path.join(BASE_DIR, "manipulated_probe_predictions_all_layers.csv"),
        probes,
        apply_pooling=True
    )
    # run neutral
    # apply_probes(
    #     NEUTRAL_CSV,
    #     os.path.join(BASE_DIR, "neutral_probe_predictions.csv"),
    #     probes,apply_pooling=True
    # )

    # Optional: Print summary statistics
    print("\n" + "=" * 50)
    print("SUMMARY STATISTICS")
    print("=" * 50)

    # Layer-wise accuracy (if you have ground truth)
    if 'emotion' in manip_results.columns:
        layer_acc = manip_results.groupby('layer').apply(
            lambda x: (x['pred'] == x['emotion']).mean()
        ).sort_values(ascending=False)

        print("\nTop 10 layers by accuracy:")
        print(layer_acc.head(10))

        print("\nBottom 10 layers by accuracy:")
        print(layer_acc.tail(10))

    # Confidence statistics
    layer_conf = manip_results.groupby('layer')['confidence'].agg(['mean', 'std', 'count'])
    print("\nConfidence statistics by layer:")
    print(layer_conf.head(10))

# =========================
# RUN MAIN
# =========================
if __name__ == "__main__":
    main()


In [ ]:
import pandas as pd
import numpy as np
import os

# =========================
# CONFIG
# =========================
# MANIP_CSV = "/content/drive/MyDrive/probing/full_probe_flamingo/abla/manipulation_probe_predictions_all_layers.csv"
# NEUTRAL_CSV = "/content/drive/MyDrive/probing/full_probe_flamingo/abla/neutral_probe_predictions.csv"

# OUTPUT_CSV = "/content/drive/MyDrive/probing/full_probe_flamingo/abla/probe_shift_anchor.csv"
MANIP_CSV = "/content/drive/MyDrive/probing/full_probe/albe/manipulated_probe_predictions_all_layers.csv"

NEUTRAL_CSV = "/content/drive/MyDrive/probing/full_probe/albe/neutral_probe_predictions.csv"

OUTPUT_CSV = "/content/drive/MyDrive/probing/full_probe/albe/probe_shift_anchor.csv"

MODEL_NAME = "flamingo"  # change if needed

EMOS = ["ANG","DIS","FEA","HAP","SAD","NEU","SUR"]

# =========================
# LOAD DATA
# =========================
manip_df = pd.read_csv(MANIP_CSV)

neutral_df = pd.read_csv(NEUTRAL_CSV)

# clean paths (important)
manip_df["neutral_path"] = manip_df["neutral_path"].astype(str).str.strip()
neutral_df["neutral_path"] = neutral_df["neutral_path"].astype(str).str.strip()

# =========================
# SAFETY CHECK
# =========================
print("Manip rows:", len(manip_df))
print("Neutral rows:", len(neutral_df))

# =========================
# MERGE (ONE-TO-MANY SAFE)
# =========================
merged = pd.merge(
    manip_df,
    neutral_df,
    on=["neutral_path", "layer"],
    suffixes=("_manip", "_neutral")
)

print("Merged rows:", len(merged))
print("Should equal manip rows (or very close)")
print(merged)
# =========================
# COMPUTE PROBE SHIFT
# =========================
results = []

for idx, row in merged.iterrows():

    entry = {
        "model": MODEL_NAME,
        "neutral_path": row["neutral_path"],
        "manipulated_path": row["manipulated_path_manip"],
        "layer": row["layer"],
    }

    # =========================
    # 1. get distributions
    # =========================
    try:
        p_n = np.array([row[f"p_{e}_neutral"] for e in EMOS])
        p_m = np.array([row[f"p_{e}_manip"] for e in EMOS])
    except KeyError:
        continue

    # =========================
    # 2. neutral anchor emotion
    # =========================
    neutral_idx = np.argmax(p_n)
    neutral_emo = EMOS[neutral_idx]

    entry["neutral_anchor"] = neutral_emo
    entry["neutral_conf"] = p_n[neutral_idx]

    # =========================
    # 3. anchor shift (CORE)
    # =========================
    entry["delta_anchor"] = p_m[neutral_idx] - p_n[neutral_idx]

    # =========================
    # 4. full per-emotion shift
    # =========================
    for i, emo in enumerate(EMOS):
        entry[f"delta_{emo}"] = p_m[i] - p_n[i]

    # =========================
    # 5. overall magnitude
    # =========================
    entry["probe_shift_l2"] = np.linalg.norm(p_m - p_n)

    # =========================
    # 6. entropy change (optional but useful)
    # =========================
    def entropy(p):
        p = np.clip(p, 1e-8, 1.0)
        return -np.sum(p * np.log(p))

    entry["entropy_neutral"] = entropy(p_n)
    entry["entropy_manip"] = entropy(p_m)
    entry["delta_entropy"] = entry["entropy_manip"] - entry["entropy_neutral"]

    results.append(entry)

# =========================
# SAVE
# =========================
out_df = pd.DataFrame(results)

out_df.to_csv(OUTPUT_CSV, index=False)

print(f"\nSaved to: {OUTPUT_CSV}")
print("Total rows:", len(out_df))

In [ ]:

#OUTPUT_CSV = "/content/drive/MyDrive/probing/full_probe_flamingo/abla/probe_shift_layerwise.csv"
OUTPUT_CSV = "/content/drive/MyDrive/probing/full_probe/albe/probe_shift_layerwise.csv"

# =========================
# LOAD
# =========================
manip_df = pd.read_csv(MANIP_CSV)
neutral_df = pd.read_csv(NEUTRAL_CSV)

# clean paths
manip_df["neutral_path"] = manip_df["neutral_path"].astype(str).str.strip()
neutral_df["neutral_path"] = neutral_df["neutral_path"].astype(str).str.strip()

# =========================
# MERGE
# =========================
merged = pd.merge(
    manip_df,
    neutral_df,
    on=["neutral_path", "layer"],
    suffixes=("_manip", "_neutral")
)

print("Merged rows:", len(merged))

# =========================
# EXTRACT GT FROM NEUTRAL PATH
# =========================
def extract_gt(path):
    if not isinstance(path, str):
        return None
    for emo in EMOS:
        if emo in path:
            return emo
    return None

merged["gt"] = merged["neutral_path"].apply(extract_gt)

# =========================
# BUILD LAYER-WISE STRUCTURE
# =========================

# group by sample pair
group_cols = ["neutral_path", "manipulated_path_manip"]

grouped = merged.groupby(group_cols)

final_rows = []

for (neutral_path, manipulated_path), group in grouped:

    row_dict = {
        "model": MODEL_NAME,
        "neutral_path": neutral_path,
        "manipulated_path": manipulated_path,
        "gt": group["gt"].iloc[0]
    }

    for _, r in group.iterrows():

        layer = r["layer"]

        try:
            p_n = np.array([r[f"p_{e}_neutral"] for e in EMOS])
            p_m = np.array([r[f"p_{e}_manip"] for e in EMOS])
        except KeyError:
            continue

        # =========================
        # baseline from neutral (per-layer!)
        # =========================
        idx = np.argmax(p_n)
        baseline_emo = EMOS[idx]

        # =========================
        # shift along that axis
        # =========================
        delta = p_m[idx] - p_n[idx]

        # save
        row_dict[f"{layer}_baseline"] = baseline_emo
        row_dict[f"{layer}_shift"] = delta

    final_rows.append(row_dict)

# =========================
# SAVE
# =========================
final_df = pd.DataFrame(final_rows)

final_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved to: {OUTPUT_CSV}")
print("Shape:", final_df.shape)

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# =========================
# CONFIG (MULTI-MODEL)
# =========================
MODELS = {
    "qwen": {
        "MANIP_CSV": "/content/drive/MyDrive/probing/full_probe/albe/manipulated_probe_predictions_all_layers.csv",
        "NEUTRAL_CSV": "/content/drive/MyDrive/probing/full_probe/albe/neutral_probe_predictions.csv",
    },
    "flamingo": {
        "MANIP_CSV": "/content/drive/MyDrive/probing/full_probe_flamingo/abla/manipulation_probe_predictions_all_layers.csv",
        "NEUTRAL_CSV": "/content/drive/MyDrive/probing/full_probe_flamingo/abla/neutral_probe_predictions.csv",
    }
}


OUTPUT_CSV = "/content/drive/MyDrive/probing/full_probe/combined_probe_shift.csv"

EMOS = ["ANG","DIS","FEA","HAP","SAD","NEU","SUR"]

# =========================
# HELPERS
# =========================
def extract_gt(path):
    if not isinstance(path, str):
        return None
    for emo in EMOS:
        if emo in path:
            return emo
    return None

def infer_task(rep_path):
    if not isinstance(rep_path, str):
        return "other"
    if "intensity" in rep_path.lower():
        return "task2"  # intensity
    else:
        return "task1"  # cue

def entropy(p):
    p = np.clip(p, 1e-8, 1.0)
    return -np.sum(p * np.log(p))

def extract_layer_id(layer):
    try:
        return int(layer.split("_")[-1])
    except:
        return -1

# =========================
# PROCESS ONE MODEL
# =========================
def process_model(model_name, manip_csv, neutral_csv):

    manip_df = pd.read_csv(manip_csv)
    neutral_df = pd.read_csv(neutral_csv)

    manip_df["neutral_path"] = manip_df["neutral_path"].astype(str).str.strip()
    neutral_df["neutral_path"] = neutral_df["neutral_path"].astype(str).str.strip()

    merged = pd.merge(
        manip_df,
        neutral_df,
        on=["neutral_path", "layer"],
        suffixes=("_manip", "_neutral")
    )

    results = []

    for _, row in merged.iterrows():

        try:
            p_n = np.array([row[f"p_{e}_neutral"] for e in EMOS])
            p_m = np.array([row[f"p_{e}_manip"] for e in EMOS])
        except KeyError:
            continue

        # =========================
        # Predictions
        # =========================
        neutral_idx = np.argmax(p_n)
        neutral_emo = EMOS[neutral_idx]

        manip_idx = np.argmax(p_m)
        manip_emo = EMOS[manip_idx]

        # =========================
        # Labels
        # =========================
        gt = extract_gt(row["neutral_path"])
        task = infer_task(row["rep_path_manip"])

        # =========================
        # Accuracy (FIXED)
        # =========================
        acc_neutral = 1.0 if gt is not None and neutral_emo == gt else 0.0
        acc_manip   = 1.0 if gt is not None and manip_emo == gt else 0.0

        entry = {
            "model": model_name,
            "neutral_path": row["neutral_path"],
            "manipulated_path": row["manipulated_path_manip"],
            "layer": row["layer"],
            "layer_id": extract_layer_id(row["layer"]),

            # labels
            "baseline_pred": neutral_emo,
            "manip_pred": manip_emo,
            "gt": gt,
            "task": task,

            # accuracy shift
            "accuracy_neutral": acc_neutral,
            "accuracy_manip": acc_manip,
            "accuracy_shift": acc_manip - acc_neutral,

            # core metrics
            "conf_shift": p_m[neutral_idx] - p_n[neutral_idx],
            "probe_shift_l2": np.linalg.norm(p_m - p_n),
            "entropy_neutral": entropy(p_n),
            "entropy_manip": entropy(p_m),
            "entropy_shift": entropy(p_m) - entropy(p_n),
        }

        results.append(entry)

    return pd.DataFrame(results)

# =========================
# RUN ALL MODELS
# =========================
all_dfs = []

for model_name, paths in MODELS.items():
    print(f"Processing {model_name}...")
    df_model = process_model(
        model_name,
        paths["MANIP_CSV"],
        paths["NEUTRAL_CSV"]
    )
    all_dfs.append(df_model)

df = pd.concat(all_dfs, ignore_index=True)

# keep only cue & intensity
df = df[df["task"].isin(["task1", "task2"])]

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved combined CSV: {OUTPUT_CSV}")

# =========================
# VISUALIZATION (2x3 GRID)
# =========================
metrics = ["conf_shift", "entropy_shift", "accuracy_shift"]
metric_titles = [
    "Confidence Shift (Δ)",
    "Entropy Shift (Δ)",
    "Accuracy Shift (Δ)"
]

models = df["model"].unique()
tasks = ["task1", "task2"]

task_names = {
    "task1": "Cue",
    "task2": "Intensity"
}

colors = {
    "task1": "#1f77b4",
    "task2": "#ff7f0e"
}

fig, axes = plt.subplots(len(models), 3, figsize=(16, 5 * len(models)), sharex=True)

if len(models) == 1:
    axes = np.expand_dims(axes, axis=0)

for i, model in enumerate(models):
    for j, metric in enumerate(metrics):

        ax = axes[i, j]
        sub_df = df[df["model"] == model]

        for task in tasks:
            task_df = sub_df[sub_df["task"] == task]

            agg = (
                task_df.groupby("layer_id")[metric]
                .agg(["mean", "std"])
                .reset_index()
                .sort_values("layer_id")
            )

            ax.plot(
                agg["layer_id"],
                agg["mean"],
                label=task_names[task],
                color=colors[task],
                linewidth=2
            )

            ax.fill_between(
                agg["layer_id"],
                agg["mean"] - agg["std"],
                agg["mean"] + agg["std"],
                color=colors[task],
                alpha=0.2
            )

        if i == 0:
            ax.set_title(metric_titles[j], fontsize=12)

        if j == 0:
            ax.set_ylabel(model.upper(), fontsize=12)

        ax.set_xlabel("Layer")
        ax.grid(alpha=0.3)

# shared legend
handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=2)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

In [ ]:
import os
import pandas as pd
from tqdm import tqdm
import torch
import librosa
import numpy as np
import shutil
from torch.utils.data import DataLoader, Dataset

from transformers import Wav2Vec2Model, Wav2Vec2FeatureExtractor

# ============================================================
# CONFIG
# ============================================================
BASE_DIR = "/content/drive/MyDrive/subprobe"
LOCAL_BASE = "/content/subprobe_local"

CSV_PATH = os.path.join(BASE_DIR, "merged_subtle_emotion.csv")
SAVE_DIR = os.path.join(LOCAL_BASE, "wav2vec_reference")
os.makedirs(SAVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 16  # Adjust based on GPU memory

print(f"Using device: {DEVICE}")
print(f"Batch size: {BATCH_SIZE}")

# ============================================================
# COPY AUDIO FILES LOCALLY
# ============================================================
def copy_audio_locally(df, source_base, target_base):
    """
    Copy all audio files to local storage for faster access
    """
    local_paths = {}

    # Create audio directory locally
    local_audio_dir = os.path.join(target_base, "audio_files")
    os.makedirs(local_audio_dir, exist_ok=True)

    # Get unique audio paths
    unique_paths = df["audio_path"].unique()

    print(f"Copying {len(unique_paths)} unique audio files to local storage...")

    for path in tqdm(unique_paths, desc="Copying audio files"):
        # Resolve original path
        orig_path = str(path).replace("\\", "/").strip()
        if not orig_path.startswith("/content/"):
            orig_path = os.path.join(source_base, orig_path)

        if not os.path.exists(orig_path):
            print(f"Warning: {orig_path} not found")
            continue

        # Create local path
        filename = os.path.basename(orig_path)
        local_path = os.path.join(local_audio_dir, filename)

        # Copy if not already exists
        if not os.path.exists(local_path):
            shutil.copy2(orig_path, local_path)

        local_paths[path] = local_path

    print(f"Copied {len(local_paths)} files to {local_audio_dir}")
    return local_paths

# ============================================================
# DATASET CLASS
# ============================================================
class AudioDataset(Dataset):
    def __init__(self, df, local_path_map):
        self.df = df
        self.local_paths = local_path_map
        self.valid_indices = []

        # Pre-filter valid files
        for idx, row in df.iterrows():
            audio_path = row["audio_path"]
            if audio_path in self.local_paths:
                self.valid_indices.append(idx)

        print(f"Found {len(self.valid_indices)} valid samples out of {len(df)}")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        actual_idx = self.valid_indices[idx]
        row = self.df.iloc[actual_idx]

        wav_path = self.local_paths[row["audio_path"]]

        # Load audio (returns numpy array)
        speech, sr = librosa.load(wav_path, sr=16000)

        return {
            "idx": actual_idx,
            "speech": speech,
            "language": row["language"],
            "labels": row["labels"],
            "label_counts": row["label_counts"],
            "original_path": row["audio_path"],
            "local_path": wav_path
        }

def collate_fn(batch):
    """
    Custom collate function to handle variable length audio
    """
    speeches = [item["speech"] for item in batch]
    indices = [item["idx"] for item in batch]
    languages = [item["language"] for item in batch]
    labels = [item["labels"] for item in batch]
    label_counts = [item["label_counts"] for item in batch]
    original_paths = [item["original_path"] for item in batch]
    local_paths = [item["local_path"] for item in batch]

    return {
        "idx": indices,
        "speech": speeches,
        "language": languages,
        "labels": labels,
        "label_counts": label_counts,
        "original_path": original_paths,
        "local_path": local_paths
    }

# ============================================================
# LOAD WAV2VEC
# ============================================================
print("\nLoading Wav2Vec2 model...")
model = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-large").to(DEVICE)
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained("facebook/wav2vec2-large")
model.eval()
print("Model loaded successfully!")

# ============================================================
# LOAD DATA
# ============================================================
print("\nLoading CSV...")
df = pd.read_csv(CSV_PATH)
print(f"Total samples in CSV: {len(df)}")

# Copy audio files locally
local_path_map = copy_audio_locally(df, BASE_DIR, LOCAL_BASE)

# Create dataset and dataloader
dataset = AudioDataset(df, local_path_map)
dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2  # Parallel loading
)

# ============================================================
# BATCH EXTRACTION FUNCTION
# ============================================================
def batch_extract_embeddings(dataloader, model, feature_extractor, device, save_dir):
    """
    Extract embeddings in batches for faster processing
    """
    all_metadata = []
    successful = 0
    failed = 0

    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Extracting embeddings")):
        speeches = batch["speech"]

        # Skip empty batch
        if not speeches:
            continue

        try:
            # Process batch through feature extractor
            inputs = feature_extractor(
                speeches,
                sampling_rate=16000,
                return_tensors="pt",
                padding=True
            )

            # Move to device
            inputs = {k: v.to(device) for k, v in inputs.items()}

            # Forward pass
            with torch.no_grad():
                outputs = model(**inputs)
                embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()

            # Save each embedding in the batch
            for i, emb in enumerate(embeddings):
                idx = batch["idx"][i]

                # Save embedding
                rep_path = os.path.join(save_dir, f"{idx}.npy")
                np.save(rep_path, emb)

                # Store metadata
                all_metadata.append({
                    "id": idx,
                    "audio_path": batch["original_path"][i],
                    "local_audio_path": batch["local_path"][i],
                    "language": batch["language"][i],
                    "labels": batch["labels"][i],
                    "label_counts": batch["label_counts"][i],
                    "rep_path": rep_path
                })
                successful += 1

        except Exception as e:
            print(f"\nError processing batch {batch_idx}: {e}")
            failed += len(speeches)
            continue

    print(f"\n✅ Successfully processed: {successful}")
    print(f"❌ Failed: {failed}")

    return all_metadata

# ============================================================
# RUN BATCH EXTRACTION
# ============================================================
print("\nStarting batch embedding extraction...")
metadata = batch_extract_embeddings(
    dataloader,
    model,
    feature_extractor,
    DEVICE,
    SAVE_DIR
)

# ============================================================
# SAVE RESULTS
# ============================================================
print("\nSaving results...")

# Create dataframe
out_df = pd.DataFrame(metadata)

# Save CSV locally
csv_local = os.path.join(SAVE_DIR, "wav2vec_reference.csv")
out_df.to_csv(csv_local, index=False)
print(f"✅ Saved locally: {csv_local}")

# Also save to Drive for persistence
drive_save_dir = os.path.join(BASE_DIR, "wav2vec_reference")
os.makedirs(drive_save_dir, exist_ok=True)
csv_drive = os.path.join(drive_save_dir, "wav2vec_reference.csv")
out_df.to_csv(csv_drive, index=False)
print(f"✅ Saved to Drive: {csv_drive}")

# Summary
print("\n" + "="*50)
print("EXTRACTION SUMMARY")
print("="*50)
print(f"Total samples processed: {len(metadata)}")
print(f"Embeddings saved to: {SAVE_DIR}")
print(f"CSV saved to: {csv_local}")
print("\nFirst few rows:")
print(out_df.head())

# Optional: Verify a sample embedding
if len(metadata) > 0:
    sample_emb = np.load(metadata[0]["rep_path"])
    print(f"\nSample embedding shape: {sample_emb.shape}")
    print(f"Sample embedding dtype: {sample_emb.dtype}")